In [12]:
import numpy as np
import pandas as pd
import rioxarray 
import rasterio as rio
from rasterio.plot import show
from rasterio.enums import Resampling
import glob
import os
from osgeo import gdal
from pathlib import Path
from scipy.ndimage import median_filter
from datetime import datetime


#from win32comext.axscript.client.framework import profile



In [5]:
LiDAR_dir = "C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR"
raster_files = glob.glob(os.path.join(LiDAR_dir, '*.tif'))

In [24]:
#See stats for rasters
# Create dataframe of mean, max, min, SD, na count

df = []
for raster in raster_files:
    with rio.open(raster) as src:
            data = src.read(1, masked=True) # masked array handles nodata
            name = os.path.basename(raster)
            parts = name.split("_")
            date_str = parts[3]        # '20230405'
            date = datetime.strptime(date_str, "%Y%m%d").date()
            # total_cells = data.size                 # total pixels
            # nan_count = data.mask.sum()              # nodata pixels
            # valid_count = total_cells - nan_count            
           
            df.append({
                "raster": date,
                "min": float(data.min()),
                "max": float(data.max()),
                "mean": float(data.mean()),
                "std": float(data.std()),
                # "nan_count": int(nan_count),
                # "valid_cells": int(valid_count),
                # "nan_fraction": nan_count / total_cells,
            })
            
df = pd.DataFrame(df)

# Optional: save to CSV
df.to_csv(os.path.join(LiDAR_dir, "raster_stats.csv"), index=False)

print(df)

        raster        min        max      mean       std  nan_count  \
0   2022-12-08 -23.840698  16.818481  0.940349  0.315307  127604047   
1   2023-02-09 -19.336304  18.437744  1.630203  0.506498  129881172   
2   2023-03-16 -17.077148  35.241699  2.464226  0.743998  132083497   
3   2023-04-05 -23.262939  38.887207  2.706930  0.855918  149289571   
4   2023-12-28 -17.476562  34.964233  0.627988  0.350654  136141663   
5   2024-01-15 -23.192383  26.385254  1.114397  0.356214  145715848   
6   2024-02-13 -27.413940  27.248291  0.869934  1.662692  201079005   
7   2024-03-15 -17.306152  28.422852  1.842306  0.572888  154066323   
8   2024-04-18 -26.150146  24.617310  1.202956  0.739757  159770529   
9   2025-01-13 -18.276001  33.880493  1.676494  0.570334  146625293   
10  2025-01-29 -16.618408  27.531738  1.341038  0.453065  155401617   
11  2025-04-04 -18.386353  28.154907  2.304307  0.851767  157123213   
12  2025-05-01 -27.215332  28.404053  1.220852  0.872216  151161227   

    v

### To begin, I will simply re-sample our 0.5-m rasters. I am preserving this code so I have an example of the GDAL structure

In [18]:
##Using GDAL
extent = (601558.000, 4862467.500, 609431.500, 4870872.500)
for raster in raster_files:
    out_dir = "C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR/resampled/cubic"
    name, ext = os.path.splitext(os.path.basename(raster))
    out_raster = os.path.join(out_dir, f"{name}_100{ext}")
    gdal.Warp(
        out_raster, 
        raster, 
        dstNodata=float("nan"),
        xRes=100,
        yRes=100,
        resampleAlg="cubic",
        outputBounds=extent)

In [25]:
# Create dataframe of mean, max, min, SD, na count
out_dir = "C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR/resampled/cubic"

df = []
for raster in glob.glob(os.path.join(out_dir, '*.tif')):
    with rio.open(raster) as src:
            data = src.read(1, masked=True) # masked array handles nodata
            name = os.path.basename(raster)
            parts = name.split("_")
            date_str = parts[3]        # '20230405'
            date = datetime.strptime(date_str, "%Y%m%d").date()
            # total_cells = data.size                 # total pixels
            # nan_count = data.mask.sum()              # nodata pixels
            # valid_count = total_cells - nan_count    # valid pixels
            
            df.append({
                "raster": date,
                "min": float(data.min()),
                "max": float(data.max()),
                "mean": float(data.mean()),
                "std": float(data.std()),
                # "nan_count": int(nan_count),
                # "valid_cells": int(valid_count),
                # "nan_fraction": nan_count / total_cells,
                
            })
            
df = pd.DataFrame(df)

# Optional: save to CSV
df.to_csv(os.path.join(out_dir, "raster_stats_100m_cubic.csv"), index=False)

print(df)

    


        raster        min       max      mean       std  nan_count  \
0   2022-12-08   0.216518  1.551489  0.940973  0.236620       3209   
1   2023-02-09   0.497737  2.751906  1.630203  0.414400       3255   
2   2023-03-16   0.834459  4.011801  2.462127  0.633169       3312   
3   2023-04-05  -0.005481  8.702649  2.706071  0.569520       3752   
4   2023-12-28  -0.155802  1.225676  0.626216  0.269890       3416   
5   2024-01-15   0.261172  1.711268  1.114940  0.232060       3659   
6   2024-02-13 -24.646212  2.203440  0.859695  1.515999       5046   
7   2024-03-15   0.592122  3.051784  1.842996  0.450203       3866   
8   2024-04-18  -0.111163  2.611377  1.204695  0.587272       4019   
9   2025-01-13   0.360956  2.555928  1.676345  0.489335       3678   
10  2025-01-29   0.377309  2.221214  1.343080  0.354175       3911   
11  2025-04-04   0.325071  3.963891  2.303230  0.736755       3950   
12  2025-05-01  -0.160170  3.062641  1.224433  0.757998       3795   

    valid_cells  na

### The same things with Rasterio, again so that I have the structure


In [ ]:
#Using Rasterio
for raster in raster_files:
    dir_name = os.path.dirname(raster)
    name, ext = os.path.splitext(os.path.basename(raster))
    out_raster = os.path.join(dir_name, f"{name}_100_rasterio{ext}")
    sd = rioxarray.open_rasterio(raster)
    sd.rio.write_nodata(np.nan, inplace=True)  # make sure nans are counted as no data
    sd_downsampled = sd.rio.reproject(
        sd.rio.crs,
        resolution = (100,100),
        resampling=Resampling.average)
    sd_downsampled.rio.to_raster(out_raster)


## We dedcided to try and rasterize our point cloud data directly. 
### Resampled the following output: ~/ice-road/results/pc-transform/laz-snow-trans_source.laz
used following pipeline on Borah:\
        [ \
    { \
        "type": "readers.las", \
        "filename": "$laz"\
    }, \
    { \
        "type": "writers.gdal", \
        "filename": "$tif", \
        "resolution": 100, \
        "output_type": "mean", \
        "dimension": "Z", \
        "data_type": "float32", \
        "nodata": -9999 \
    } \
] \
Outputs saved to C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR2

In [ ]:
# # set up directory paths
# # I will need to difference the re-rasterized point clouds from the reference, which also needs to be resampled
# 
# dir = Path("C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR2")
# tifs = glob.glob(os.path.join(dir, "*.tif"))
# ref = "C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR2/ref/MCS_REFDEM_WGS84_cubic.tif"

In [ ]:
# #I need to see if raster profiles match
# with rio.open(ref) as ref_src:
#     ref_src.read(1, masked=True)
#     print(ref_src.profile)
#     
# for raster in tifs:
#     with rio.open(raster) as src:
#         src.read(1, masked=True)
#         print(src.profile)

In [ ]:
# # Profiles do not match, so I need to resample
# # Need to pull grid info from ref
# 
# with rio.open(ref) as ref_src:
#     ref_dem = ref_src.read(1, masked=True)
#     ref_profile = ref_src.profile
#     ref_width = ref_src.width
#     ref_height = ref_src.height
#     ref_transform = ref_src.transform
# 
# #resample rasters prior to differencing 
# 
# for raster_file in tifs:
#     with rio.open(raster_file) as src:
#         # Resample to match reference dimensions
#         data = src.read(1,
#             out_shape=(ref_height, ref_width),
#             resampling=Resampling.bilinear  # DEMs usually bilinear
#         )
# 
#         # Update profile for new raster
#         out_profile = ref_profile
#         out_profile.update({
#             "dtype": data.dtype,
#             "height": ref_height,
#             "width": ref_width,
#             "transform": ref_transform,
#             "nodata": -9999
#         })
# 
#         # Write aligned raster
#         resampled_dir = os.path.join(dir, "resampled")
#         os.makedirs(resampled_dir, exist_ok=True)  # creates it if it doesn't exist
#         out_path = os.path.join(resampled_dir, os.path.basename(raster_file))
#         with rio.open(out_path, "w", **out_profile) as dst:
#             dst.write(data, 1)
# 
# print("All rasters aligned to reference raster.")

In [ ]:
# ## Difference rasters
# 
# for raster in glob.glob(os.path.join(resampled_dir, "*.tif")): 
#     with rio.open(raster) as src:
#         base = os.path.basename(raster).split("-")[0]   
#         out_raster = os.path.join(dir, f"{base}_sd.tif")
#         
#         # Read DTM and metadata
#         dtm = src.read(1, masked=True)
#         dsm_meta = src.profile
# 
#         # Compute snow depth
#         sd = dtm - ref_dem
# 
#         # Write output raster
#         with rio.open(out_raster, "w", **dsm_meta) as dst:
#             dst.write(sd.astype("float32"), 1)

### Some of these rasters deviate from the reference raster by over 100 (close to 200) m. Therefore, returning to manipulation of 0.5-m rasters
### I think there is an issue with averaging the point cloud over a 100-m region when there are data gaps 

In [27]:
#chose a single test dataset or full folder

LiDAR_dir = Path("C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR/test")
raster_files = glob.glob(os.path.join(LiDAR_dir, '*.tif'))
#raster_files = glob.glob(os.path.join(LiDAR_dir, "SNEX_MCS_Lidar_20250404_SD_V01.0.tif"))

for raster in raster_files:
    print(raster)
    #with rio.open(raster) as src:
        #sd = src.read(1, masked=True)
        #print(sd.max(), sd.min())
        #print(src.shape)
      

C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR\test\SNEX_MCS_Lidar_20250129_SD_V01.0_filtered.tif
C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR\test\SNEX_MCS_Lidar_20250404_SD_V01.0_filtered.tif
C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR\test\SNEX_MCS_Lidar_20250501_SD_V01.0_filtered.tif


In [ ]:
# Remove below 0 values and remove highest 1% of values (outliers)
# I found that most negative values are along the road or in areas of no snow or what I believe to be melt
# I want to see though if the +/- 3 standard deviations from the mean accounts for these values
# doesn't work- a lot of values are way beyond three SD at highest elevations

# for raster in raster_files:
#     with rio.open(raster) as src:
#         sd = src.read(1, masked=True).filled(np.nan)
#         mean_val = np.nanmean(sd)
#         std_dev  = np.nanstd(sd)
#         threshold = 3 * std_dev
#         print(raster)
#         print( {mean_val},{std_dev},{mean_val + threshold},{mean_val - threshold} )

In [21]:
# https://docs.scipy.org/doc/scipy/tutorial/ndimage.html#filter-functions

for raster in raster_files:
    with rio.open(raster) as src:
        out_raster = os.path.join(LiDAR_dir / "filtered", os.path.basename(raster).replace(".tif", "_filtered.tif"))
        #print(out_raster)
        #print(src.shape)
        sd = src.read(1, masked=True).filled(np.nan)  # <-- fill masked with NaN
        sd[sd < -1] = np.nan
        sd[sd > 10] = np.nan
        meta = src.profile
        # Write output raster
        with rio.open(out_raster, "w", **meta) as dst:
            dst.write(sd.astype("float32"), 1)
        
        

In [30]:
for raster in raster_files:
    with rio.open(raster) as src:       
        sd = median_filter(sd, size=10, mode='nearest')
        out_dir = os.path.join(os.path.dirname(raster), "outputs")
        out_raster = os.path.join(out_dir,os.path.basename(raster).replace(".tif", "_mw10.tif"))
             # size is number of pixels, so 20 = 10-m
        #vals = sd[np.isfinite(sd)]
        #upper = np.percentile(vals, 99.9)
        #sd[sd > upper] = np.nan
        #print(np.nanmax(sd), np.nanmin(sd))
        meta = src.profile
        # Write output raster
        with rio.open(out_raster, "w", **meta) as dst:
            dst.write(sd.astype("float32"), 1)


In [53]:
import rasterio
from rasterio.fill import fillnodata

LiDAR_dir = Path("C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR/filtered")
raster_files = glob.glob(os.path.join(LiDAR_dir, '*.tif'))
print(raster_files)
#raster_files = glob.glob(os.path.join(LiDAR_dir, "SNEX_MCS_Lidar_20250404_SD_V01.0.tif"))

# https://rasterio.readthedocs.io/en/stable/api/rasterio.fill.html#rasterio.fill.fillnodata

# raster = Path("C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR/test")
# dir_name = os.path.dirname(raster)
# name, ext = os.path.splitext(os.path.basename(raster))
# out_raster = os.path.join(dir_name, f"{name}_fill_10{ext}")
# sd = rioxarray.open_rasterio(raster)

#sd.rio.write_nodata(np.nan, inplace=True)  # make sure nans are counted as no data
# 1. Open the elevation file
for raster in raster_files:
    with rasterio.open(raster) as src:
        out_raster = os.path.join(LiDAR_dir, 'rasterio_fill_mw20', os.path.basename(raster).replace(".tif", "_mw20_smooth3.tif"))
        profile = src.profile
    # Read the first band and the mask (identifies NoData areas)
        image = src.read(1, masked=True)

# 2. Fill the gaps
# max_search_distance: max pixels to search for valid values
# smoothing_iterations: number of passes to smooth filled areas
        filled_data = fillnodata(image, max_search_distance=20, smoothing_iterations=3)
    with rasterio.open(out_raster, 'w', **profile) as dst:
        dst.write(filled_data, 1)
        

['C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\SNEX_MCS_Lidar_20221208_SD_V01.0_filtered.tif', 'C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\SNEX_MCS_Lidar_20230209_SD_V01.0_filtered.tif', 'C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\SNEX_MCS_Lidar_20230316_SD_V01.0_filtered.tif', 'C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\SNEX_MCS_Lidar_20230405_SD_V01.0_filtered.tif', 'C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\SNEX_MCS_Lidar_20231228_SD_V01.0_filtered.tif', 'C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\SNEX_MCS_Lidar_20240115_SD_V01.0_filtered.tif', 'C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\SNEX_MCS_Lidar_20240213_SD_V01.0_filtered.tif', 'C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\SNEX_MCS_Lidar_20240315_SD_V01.0_filtered.tif', 'C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\SNEX_MCS_Lidar_20240418_SD_V01.0_filtered.tif', 'C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\SNEX_MCS_Lidar_20250113_SD_

In [55]:
LiDAR_dir = Path("C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR/filtered/rasterio_fill_mw20")
raster_files = glob.glob(os.path.join(LiDAR_dir, '*.tif'))
print(raster_files)

for raster in raster_files:
    out_dir = "C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR/filtered/rasterio_fill_mw20/cubic"
    out_raster = os.path.join(out_dir,os.path.basename(raster).replace(".tif", "_cubic_100.tif"))
    gdal.Warp(
        out_raster, 
        raster, 
        dstNodata=float("nan"),
        xRes=100,
        yRes=100,
        resampleAlg="cubic",
        outputBounds=extent)

['C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\rasterio_fill_mw20\\SNEX_MCS_Lidar_20221208_SD_V01.0_filtered_mw20_smooth3.tif', 'C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\rasterio_fill_mw20\\SNEX_MCS_Lidar_20230209_SD_V01.0_filtered_mw20_smooth3.tif', 'C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\rasterio_fill_mw20\\SNEX_MCS_Lidar_20230316_SD_V01.0_filtered_mw20_smooth3.tif', 'C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\rasterio_fill_mw20\\SNEX_MCS_Lidar_20230405_SD_V01.0_filtered_mw20_smooth3.tif', 'C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\rasterio_fill_mw20\\SNEX_MCS_Lidar_20231228_SD_V01.0_filtered_mw20_smooth3.tif', 'C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\rasterio_fill_mw20\\SNEX_MCS_Lidar_20240115_SD_V01.0_filtered_mw20_smooth3.tif', 'C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\rasterio_fill_mw20\\SNEX_MCS_Lidar_20240213_SD_V01.0_filtered_mw20_smooth3.tif', 'C:\\Users\\RDCRLSMC\\Desktop\\SIRO\\LiDAR\\filtered\\

In [39]:
# #Resample rasters to 100-m
# 
# upscale_factor = 200
# 
# for raster in glob.glob(os.path.join(LiDAR_dir / "filtered", "*.tif")):
#     dir_name = os.path.dirname(raster)
#     name, ext = os.path.splitext(os.path.basename(raster))
#     out_raster = os.path.join(dir_name, f"{name}_100_rasterio{ext}")
#     sd = rioxarray.open_rasterio(raster)
#     sd.rio.write_nodata(np.nan, inplace=True)  # make sure nans are counted as no data
#     sd_downsampled = sd.rio.reproject(
#         sd.rio.crs,
#         resolution = (100,100),
#         resampling=Resampling.average)
#     sd_downsampled.rio.to_raster(out_raster)        